In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

In [2]:
class Conv33:

    def __init__(self, n_filters):
        self.n_filters = n_filters
        self.filters = np.random.randn(n_filters, 3, 3)/9

    def iterate_regions(self, image):
        h, w = image.shape
        for i in range(h-2):
            for j in range(w-2):
                image_region = image[i:(i+3), j:(j+3)]
                yield image_region, i, j

    def forward(self, input):
        self.last_input = input
        h, w = input.shape
        output = np.zeros((h, w, self.n_filters))
        for image_region, i, j in self.iterate_regions(input):
            output[i, j] = np.sum(image_region*self.filters, axis=(1,2))
        return output

    def backward(self, grad_out, learn_rate):
        d_l_d_filters = np.zeros(self.filters.shape)
        for image_region, i, j in self.iterate_regions(self.last_input):
            for f in range(self.n_filters):
                d_l_d_filters[f] = image_region * grad_out[i, j, f]
        self.filters -= learn_rate * d_l_d_filters
        return None

In [3]:
class MaxPool2:
    def iterate_region(self, image):
        h, w, _ = image.shape
        new_h = h//2
        new_w = w//2
        for i in range(new_h):
            for j in range(new_w):
                image_region = image[(i*2) : ((i*2)+2), (j*2) : ((j*2)+2)]
                yield image_region, i, j

    def forward(self, input):
        self.last_input = input
        h, w, n_filters = input.shape
        output = np.zeros((h//2, w//2, n_filters))
        for image_region, i, j in self.iterate_region(input):
            output[i, j] = np.amax(image_region, axis=(0,1))
        return output

    def backward(self, grad_out, learn_rate):
        d_l_d_input = np.zeros(self.last_input.shape)
        for image_region, i, j in self.iterate_region(self.last_input):
            h, w, f = image_region.shape
            amax = np.amax(image_region, axis=(0, 1))
            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        if image_region[i2, j2, f2] == amax[f2]:
                            d_l_d_input[i2 * 2 + i2, j2 * 2 + j2, f2] = grad_out[i2, j2, f2]
        return d_l_d_input

In [4]:
class SoftMax:

    def __init__(self, input_size, nodes):
        self.weights = np.random.randn(input_size, nodes)/input_size
        self.bias = np.zeros((nodes))

    def forward(self, input):
        self.last_input_shape = input.shape
        input = input.flatten()
        self.last_input = input
        input_size, node = self.weights.shape
        total = np.dot(input, self.weights) + self.bias
        self.last_total = total
        exp = np.exp(total)
        return exp/np.sum(exp, axis=0)

    def backward(self, grad_out, learn_rate):
        for i, grad in enumerate(grad_out):
            if grad == 0:
                continue
            t_exp = np.exp()
            exp_sum = np.sum(t_exp)
            d_out_d_t = -t_exp[i] * t_exp / (exp_sum ** 2)
            d_out_d_t[i] = -t_exp[i] * (exp_sum - t_exp[i]) / (exp_sum ** 2)
            d_t_d_w = self.last_input
            d_t_d_b = 1
            d_t_d_input = self.weights
            d_l_d_t = grad * d_out_d_t
            d_l_d_w = d_l_d_t[np.newaxis].T @ d_t_d_w[np.newaxis]
            d_l_d_b = d_l_d_t * d_t_d_b
            d_l_d_input = d_l_d_t * d_t_d_input
            self.weights -= learn_rate * d_l_d_w
            self.bias -= learn_rate * d_l_d_b
            return d_l_d_input.reshape(self.last_input_shape)